In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark= SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
from pyspark.sql.functions import*

In [3]:
loan_def_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema",True) \
.load("/user/itv024771/landing_club/loan_defaulter")

In [4]:
loan_def_df.show()

+--------------------+--------------+---------+------------+---------------+------------+-----------+-----------+-------+--------------------+------------------+----------------------+
|              mem_id|inq_last_6mths|out_prncp|last_pymnt_d|last_pymnt_amnt|tot_coll_amt|delinq_2yrs|delinq_amnt|pub_rec|pub_rec_bankruptcies|total_rec_late_fee|mths_since_last_delinq|
+--------------------+--------------+---------+------------+---------------+------------+-----------+-----------+-------+--------------------+------------------+----------------------+
|f637eae17177579c9...|             0|  11111.8|    Feb-2019|         378.59|         198|          0|          0|      0|                   0|               0.0|                    71|
|f784f92e45f79ac13...|             0|  4651.97|    Feb-2019|         193.13|           0|          0|          0|      0|                   0|               0.0|                    26|
|8d2d1611cc89e54b0...|             0| 18451.49|    Feb-2019|          408.7

In [5]:
loan_def_df.printSchema()

root
 |-- mem_id: string (nullable = true)
 |-- inq_last_6mths: integer (nullable = true)
 |-- out_prncp: double (nullable = true)
 |-- last_pymnt_d: string (nullable = true)
 |-- last_pymnt_amnt: double (nullable = true)
 |-- tot_coll_amt: integer (nullable = true)
 |-- delinq_2yrs: integer (nullable = true)
 |-- delinq_amnt: integer (nullable = true)
 |-- pub_rec: integer (nullable = true)
 |-- pub_rec_bankruptcies: integer (nullable = true)
 |-- total_rec_late_fee: double (nullable = true)
 |-- mths_since_last_delinq: integer (nullable = true)



In [6]:
loan_defaulter_df = loan_def_df.withColumn("ingest_date", current_timestamp())

In [7]:
loan_defaulter_df.show()

+--------------------+--------------+---------+------------+---------------+------------+-----------+-----------+-------+--------------------+------------------+----------------------+--------------------+
|              mem_id|inq_last_6mths|out_prncp|last_pymnt_d|last_pymnt_amnt|tot_coll_amt|delinq_2yrs|delinq_amnt|pub_rec|pub_rec_bankruptcies|total_rec_late_fee|mths_since_last_delinq|         ingest_date|
+--------------------+--------------+---------+------------+---------------+------------+-----------+-----------+-------+--------------------+------------------+----------------------+--------------------+
|f637eae17177579c9...|             0|  11111.8|    Feb-2019|         378.59|         198|          0|          0|      0|                   0|               0.0|                    71|2026-03-29 08:03:...|
|f784f92e45f79ac13...|             0|  4651.97|    Feb-2019|         193.13|           0|          0|          0|      0|                   0|               0.0|               

In [8]:
loan_defaulter_df.createOrReplaceTempView("defaulters")

In [9]:
spark.sql("select distinct(delinq_2yrs), count(*) as total_delinq from defaulters group by delinq_2yrs order by total_delinq desc").show(50)

+-----------+------------+
|delinq_2yrs|total_delinq|
+-----------+------------+
|          0|      188753|
|          1|       22817|
|          2|        5625|
|          3|        1936|
|          4|         851|
|          5|         417|
|          6|         238|
|          7|         116|
|          8|          70|
|          9|          52|
|         10|          33|
|         11|          22|
|         12|          19|
|         13|          11|
|         14|          10|
|         15|           8|
|         16|           7|
|         19|           2|
|         24|           2|
|         17|           2|
|       null|           1|
|         23|           1|
|         35|           1|
|         21|           1|
|         18|           1|
|         58|           1|
+-----------+------------+



In [10]:
Adding this null column to 0:

SyntaxError: invalid syntax (<ipython-input-10-352404d6c1bc>, line 1)

In [11]:
loan_def_upd_df = loan_defaulter_df.withColumn("delinq_2yrs",col("delinq_2yrs")).fillna(0, subset=["delinq_2yrs"])

In [12]:
loan_def_upd_df.createOrReplaceTempView("defaulters")

In [13]:
spark.sql("select distinct(delinq_2yrs), count(*) as total_delinq from defaulters group by delinq_2yrs order by total_delinq desc").show(30)

+-----------+------------+
|delinq_2yrs|total_delinq|
+-----------+------------+
|          0|      188754|
|          1|       22817|
|          2|        5625|
|          3|        1936|
|          4|         851|
|          5|         417|
|          6|         238|
|          7|         116|
|          8|          70|
|          9|          52|
|         10|          33|
|         11|          22|
|         12|          19|
|         13|          11|
|         14|          10|
|         15|           8|
|         16|           7|
|         24|           2|
|         19|           2|
|         17|           2|
|         35|           1|
|         21|           1|
|         23|           1|
|         58|           1|
|         18|           1|
+-----------+------------+



In [17]:
spark.sql("select count(*) from defaulters")

count(1)
220997


In [ ]:
Now I'm create a seperate df for deliquenscies:

In [15]:
loan_def_delinq_df = spark.sql("select mem_id, delinq_2yrs, delinq_amnt, mths_since_last_delinq from defaulters where delinq_2yrs > 0")

In [16]:
loan_def_delinq_df.show()

+--------------------+-----------+-----------+----------------------+
|              mem_id|delinq_2yrs|delinq_amnt|mths_since_last_delinq|
+--------------------+-----------+-----------+----------------------+
|11bfcc1d0efcaee58...|          4|          0|                    12|
|ef79afb631346427d...|          1|          0|                    22|
|ddfcbcc8afd62eaa3...|          1|          0|                    15|
|5c5f858adfc9d275c...|          1|          0|                    20|
|3da152dfeff187fec...|          1|          0|                    12|
|cf0075afa37c95cd0...|          2|          0|                    20|
|04af26e2a2b2be471...|          3|          0|                    23|
|c6f7996af48e7e43d...|          1|          0|                    10|
|cc0dd2eff0068a9e5...|          1|          0|                     5|
|3e0d7a064bc0b587e...|          2|          0|                     4|
|99bab817bd80f7fd1...|          1|          0|                    19|
|1b9aa6261571e1dde..

In [18]:
loan_def_delinq_df.count()

32243

In [19]:
loan_def_delinq_df.createOrReplaceTempView("loan_def_delinq")

In [21]:
loan_def_delinq_df.repartition(1).write \
.format("parquet") \
.option("header", True) \
.mode("overwrite") \
.option("path", "/user/itv024771/landing_club/cleaned/loan_def_delinq") \
.save()

In [22]:
loan_def_upd_df.repartition(1).write \
.format("parquet") \
.option("header", True) \
.mode("overwrite") \
.option("path", "/user/itv024771/landing_club/cleaned/loan_defaulters") \
.save()